#### Setup

In [ ]:
# Using pip
# %pip install torch==2.1.2 --index-url https://download.pytorch.org/whl/cu121

# Using conda
%conda install -y -c pytorch -c nvidia pytorch=2.1.2 pytorch-cuda=12.1

In [ ]:
# HuggingFace ecosystem
%pip install transformers==4.40.2 datasets accelerate -U

# Utilities
%pip install scikit-learn python-dotenv pyarrow numpy==1.26.4 ipywidgets==7.8.5 -U

##### Import packages, set device and label maps

In [ ]:
import notebook_init  # Set visible GPUs
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from pathlib import Path
from tqdm.auto import tqdm
from datasets import Dataset

from utils import load_model, load_tokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

LABEL_ID = {
    "HUMAN_GENERATED": 0,
    "MACHINE_GENERATED": 1
}

LABEL_TEXT = {
    0: "HUMAN_GENERATED",
    1: "MACHINE_GENERATED"
}


##### Change to root for accessing data directory

In [ ]:
import os

current_dir = Path.cwd()

if current_dir.name == "notebooks" and (
    Path.exists(current_dir.parent / Path("configs/fine_tune"))
    and Path.exists(current_dir.parent / Path("data"))
):
    os.chdir(current_dir.parent)
    print(f"Current directory: {Path.cwd()}")
elif current_dir.name == "dt133g-thesis-project":
    print(f"Current directory: {Path.cwd()}")
else:
    print("Ensure configs and data exists before running this notebook..")

### Initialize Experiments

##### Configure run with names and paths to models and data directories

In [ ]:
pretrained_models = {
    "cb": "microsoft/codebert-base",
    "gc": "microsoft/graphcodebert-base",
    "ux": "microsoft/unixcoder-base",
    "ct": "Salesforce/codet5p-770m",
    "ds": "deepseek-ai/deepseek-coder-1.3b-base",
}
pretrained_models = {
    "cb": ("microsoft/codebert-base", "encoder"),
    "gc": ("microsoft/graphcodebert-base", "encoder"),
    "ux": ("microsoft/unixcoder-base", "encoder"),
    "ct_enc": ("Salesforce/codet5p-770m", "encoder"),
    "ct_seq": ("Salesforce/codet5p-770m", "seq2seq"),
    "ds": ("deepseek-ai/deepseek-coder-1.3b-base", "causal"),
}

FULL_MODEL_NAME, MODEL_TYPE = pretrained_models["cb"]

DATASET_NAME = "droidcollection"
SUBSET_NAME = "tier_4"

MODEL_PATH = f"data/fine_tune/models/{DATASET_NAME}/{FULL_MODEL_NAME}"
MODEL_NAME = FULL_MODEL_NAME.split("/")[-1].rsplit("-")[0]


print(f"MODEL_NAME: {MODEL_NAME}, MODEL_TYPE: {MODEL_TYPE}")

SAVE_DIR = MODEL_PATH / Path("eval_outputs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

##### Initialize dataset loader

In [ ]:
def load_dataset():
    if SUBSET_NAME == "test":
        print(f"Loading dataset: {DATASET_NAME}/test.parquet")
        test_ds = Dataset.from_parquet(
            f"data/_06_generated_splits/{DATASET_NAME}/test.parquet",
            columns=["Index", "code", "label"]
        )
        test_ds = test_ds.map(
            lambda _, idx: {"snippet_id": f"row_{idx}"},
            with_indices=True,
            remove_columns=["Index"]
        )
        return test_ds

    else:
        print(f"Loading dataset: {SUBSET_NAME}/augmented_dataset.parquet")
        return Dataset.from_parquet(
            f"data/transformations/{DATASET_NAME}/{SUBSET_NAME}/augmented_dataset.parquet",
            columns=["snippet_id", "mutated_code", "label"]
        )


##### Initialize dropout logic

In [ ]:
def init_mc_dropout(model):
    """
    Safely configures specific model architectures for MC Dropout evaluation.
    Converts only the intended structural layers into training mode.
    """
    # Freeze global state
    model.eval()
    
    activated_count = 0
    
    if MODEL_NAME in ["codebert", "graphcodebert", "unixcoder"]:
        for module in model.modules():
            if isinstance(module, torch.nn.Dropout):
                module.train()
                activated_count += 1
                
    elif MODEL_NAME == "codet5p":
        for module in model.modules():
            if isinstance(module, torch.nn.Dropout):
                module.train()
                activated_count += 1
                
    elif MODEL_NAME == "deepseek":
        from transformers.models.llama.modeling_llama import LlamaAttention

        for name, module in model.named_modules():
            if isinstance(module, torch.nn.Dropout) or "dropout" in name.lower():
                module.train()
                activated_count += 1
                
    print(f"[{MODEL_NAME.upper()}] Confirmed: Activated exactly {activated_count} specific structural layers for MC Dropout.")

##### Initialize computation logic

In [ ]:
def pre_tokenize_dataset(samples, tokenizer, model_type):
    """
    Tokenizes raw text upfront into highly compact NumPy arrays.
    """
    print(f"Pre-tokenizing {len(samples)} samples for {model_type.upper()} architecture...")
    
    # Pre-allocate flat lists for IDs and structural metrics
    snippet_ids = []
    labels = []
    
    # Store token patterns as standard arrays
    all_input_ids = []
    all_attention_masks = []
    
    for item in tqdm(samples, desc="Tokenizing Raw Code Strings"):
        raw_code = item.get("code") or item.get("mutated_code")
        
        if model_type == "causal":
            prompt = "Classify the following code(output either HUMAN_GENERATED or MACHINE_GENERATED):\n\n" + raw_code
        elif model_type == "seq2seq":
            prompt = "classify: " + raw_code
        else:
            prompt = raw_code

        # Tokenize directly to int32 NumPy format arrays
        enc = tokenizer(
            prompt, 
            truncation=True, 
            max_length=512, 
            padding="max_length", # Fixed padding here simplifies loading later
            return_tensors="np"
        )
        

        snippet_ids.append(item["snippet_id"])
        labels.append(item["label"])
        
        # Squeeze out batch dimension from tokenizer return format
        all_input_ids.append(enc["input_ids"][0].astype(np.int32))
        all_attention_masks.append(enc["attention_mask"][0].astype(np.int8))
        
    # Stack individual samples into dense, ultra-compact matrix blocks
    packed_dataset = {
        "snippet_id": snippet_ids,
        "label": np.array(labels, dtype=np.int64),
        "input_ids": np.stack(all_input_ids, axis=0),
        "attention_mask": np.stack(all_attention_masks, axis=0)
    }
    
    print("✅ Pre-tokenization complete! Memory optimized.")
    return packed_dataset


In [ ]:
NUM_PASSES = 10
P_DROP = 0.1
SCALE_CORRECTION = 1.0 - P_DROP


def compute_tier_metrics(packed_data, model, tokenizer):
    """
    Runs stochastic forward passes extracting directly from dense matrix packages.
    Calculates Mean Probabilities, Predictive Entropy, and Epistemic Variance.
    """
    print(f"\nProcessing {SUBSET_NAME.title()} Data on device: {DEVICE.upper()} | Architecture: {MODEL_TYPE.upper()}, Model: {MODEL_NAME.upper()}")

    if MODEL_TYPE == "causal":
        H_ID = tokenizer.encode("H", add_special_tokens=False)[0]
        M_ID = tokenizer.encode("M", add_special_tokens=False)[0]

    if MODEL_TYPE == "seq2seq":
        H_TOK = tokenizer(LABEL_TEXT[0], return_tensors="pt").input_ids.to(DEVICE)
        M_TOK = tokenizer(LABEL_TEXT[1], return_tensors="pt").input_ids.to(DEVICE)
        ALL_IDS = torch.cat([H_TOK, M_TOK], dim=0)
    
    flat_records = []
    model.to(DEVICE)
    init_mc_dropout(model)
    
    # VRAM scale profiles
    OPTIMIZED_BATCH = 64 if MODEL_TYPE == "encoder" else 16
    total_samples = len(packed_data["snippet_id"])
    loss_fct = torch.nn.CrossEntropyLoss(reduction="none") 

    # Manually step through batches
    for start_idx in tqdm(range(0, total_samples, OPTIMIZED_BATCH), desc=f"Evaluating {SUBSET_NAME.title()}"):
        end_idx = min(start_idx + OPTIMIZED_BATCH, total_samples)
        batch_size = end_idx - start_idx
        
        inputs = {
            "input_ids": torch.from_numpy(packed_data["input_ids"][start_idx:end_idx]).to(DEVICE),
            "attention_mask": torch.from_numpy(packed_data["attention_mask"][start_idx:end_idx]).to(DEVICE).long()
        }
        
        batch_labels = packed_data["label"][start_idx:end_idx]
        batch_snippet_ids = packed_data["snippet_id"][start_idx:end_idx]
        batch_pass_probs = None
        
        if MODEL_TYPE == "causal": 
            expanded_inputs = {k: v.repeat_interleave(NUM_PASSES, dim=0) for k, v in inputs.items()}

            with torch.amp.autocast('cuda'):
                with torch.no_grad():
                    outputs = model(**expanded_inputs)
                    
                    h_logit = outputs.logits[:, -1, H_ID] * SCALE_CORRECTION
                    m_logit = outputs.logits[:, -1, M_ID] * SCALE_CORRECTION
                    
                    logits = torch.stack([h_logit, m_logit], dim=-1)
                    batch_pass_probs = torch.softmax(logits, dim=-1).view(batch_size, NUM_PASSES, 2).cpu()

        elif MODEL_TYPE == "seq2seq": 
            total_passes = NUM_PASSES * 2
            seq2seq_inputs = {k: v.repeat_interleave(total_passes, dim=0) for k, v in inputs.items()}
            label_ids = ALL_IDS.repeat(batch_size * NUM_PASSES, 1)

            with torch.amp.autocast('cuda'):
                with torch.no_grad():
                    out = model(**seq2seq_inputs, labels=label_ids)
                    
                    per_token_loss = loss_fct(out.logits.view(-1, out.logits.size(-1)), label_ids.view(-1))
                    per_seq_loss = per_token_loss.view(batch_size * total_passes, -1).sum(dim=1)
                    
                    scaled_nll = -per_seq_loss * SCALE_CORRECTION
                    
                    nll_h = scaled_nll[0::2]
                    nll_m = scaled_nll[1::2]
                    
                    logits_2class = torch.stack([nll_h, nll_m], dim=-1)
                    batch_pass_probs = torch.softmax(logits_2class, dim=-1).view(batch_size, NUM_PASSES, 2).cpu()

        else: # Encoder
            expanded_inputs = {k: v.repeat_interleave(NUM_PASSES, dim=0) for k, v in inputs.items()}

            with torch.amp.autocast('cuda'):
                with torch.no_grad():
                    outputs = model(**expanded_inputs)
                    
                    logits = outputs.logits * SCALE_CORRECTION
                    batch_pass_probs = torch.softmax(logits, dim=-1).view(batch_size, NUM_PASSES, -1).cpu()

        # Vectorized Uncertainty Calculations
        mean_probs = batch_pass_probs.mean(dim=1)
        pred_labels = mean_probs.argmax(dim=-1)
        
        eps = 1e-9
        predictive_entropies = -torch.sum(mean_probs * torch.log(mean_probs + eps), dim=-1)
        epistemic_variances = torch.sum(batch_pass_probs.var(dim=1, unbiased=True), dim=-1)

        np_mean_probs = mean_probs.numpy()
        np_pred_labels = pred_labels.numpy()
        np_entropies = predictive_entropies.numpy()
        np_variances = epistemic_variances.numpy()

        for idx in range(batch_size):
            true_label = int(batch_labels[idx])
            pred_label = int(np_pred_labels[idx])
            
            target_idx = true_label if true_label != -1 else pred_label
            conf_target = float(np_mean_probs[idx, target_idx])

            flat_records.append({
                "model_name": MODEL_NAME,
                "model_type": MODEL_TYPE,
                "tier": SUBSET_NAME,
                "snippet_id": batch_snippet_ids[idx],
                "true_label": true_label,
                "pred_label": pred_label,
                "is_correct": int(true_label == pred_label) if true_label != -1 else None,
                "conf_target": conf_target,
                "entropy": float(np_entropies[idx]),
                "epistemic_variance": float(np_variances[idx])
            })

    print(f"── {SUBSET_NAME} Successfully Processed!\n")
    return pd.DataFrame(flat_records)


### Run experiment

##### Load tokenizer, model and dataset

In [ ]:
tokenizer = load_tokenizer(FULL_MODEL_NAME)
model = load_model(MODEL_PATH, tokenizer, DEVICE)
dataset = load_dataset()

print(dataset)

##### Compute metrics for a single dataset

In [ ]:
OUTPUT_CSV = SAVE_DIR / f"ood_metrics_{MODEL_NAME}.csv"

tokenized_samples = pre_tokenize_dataset(dataset, tokenizer, MODEL_TYPE)
result_df = compute_tier_metrics(tokenized_samples, model, tokenizer)

print(f"\n=== Summary Metrics for {SUBSET_NAME.title()} ===")
print(f"Total Evaluated:      {len(result_df)}")
if result_df["is_correct"].notna().any():
    print(f"Mean Accuracy Score:  {result_df['is_correct'].mean():.4f}")
print(f"Average Entropy:      {result_df['entropy'].mean():.4f}")
print(f"Average Variance:     {result_df['epistemic_variance'].mean():.4f}")

first_num = 5

columns = result_df[["snippet_id", "true_label", "pred_label", "conf_target", "entropy", "epistemic_variance"]][:first_num].to_string(index=False)
print(f"\n {'-'*len(' '.join(columns))}")
print(f"Metrics of first {first_num} samples:\n{columns}")
print(f"{'-'*len(' '.join(columns))}\n")

import gc
del model
if 'tokenized_samples' in locals():
    del tokenized_samples
gc.collect()
torch.cuda.empty_cache()

##### Save results

In [ ]:
if not OUTPUT_CSV.exists():
    result_df.to_csv(OUTPUT_CSV, index=False)
    print(f"-- Created new tracking repository: {OUTPUT_CSV}")
else:
    result_df.to_csv(OUTPUT_CSV, mode='a', header=False, index=False)
    print(f"-- Appended records to existing repository: {OUTPUT_CSV}")

##### Compute metrics and save results for all tiers

In [ ]:
OUTPUT_CSV = SAVE_DIR / f"ood_metrics_{MODEL_NAME}.csv"
dataset_names = ["test", "tier_1", "tier_2", "tier_3", "tier_4"]

tokenizer = load_tokenizer(FULL_MODEL_NAME)
model = load_model(MODEL_PATH, tokenizer, DEVICE)

for name in dataset_names:
    SUBSET_NAME = name
    ds = load_dataset()
    tokenized_samples = pre_tokenize_dataset(ds, tokenizer, MODEL_TYPE)
    result_df = compute_tier_metrics(tokenized_samples, model, tokenizer)

    if not OUTPUT_CSV.exists():
        result_df.to_csv(OUTPUT_CSV, index=False)
        print(f"-- Created new tracking repository: {OUTPUT_CSV}")
    else:
        result_df.to_csv(OUTPUT_CSV, mode='a', header=False, index=False)
        print(f"-- Appended records to existing repository: {OUTPUT_CSV}")